In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request

import numpy as np

# Download Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Use only 200,000 characters to keep the experiment small
text = text[:200000]

print("Dataset length:", len(text))
print(text[:500])

Dataset length: 200000
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encode text into numbers
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Encoded data:", data[:20])

Vocabulary size: 62
Encoded data: tensor([16, 44, 53, 54, 55,  1, 13, 44, 55, 44, 61, 40, 49,  8,  0, 12, 40, 41,
        50, 53])


In [5]:
# Split dataset
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

# Sequence length
block_size = 100

# Generate batches
def get_batch(split):

    source = train_data if split == "train" else val_data

    ix = torch.randint(len(source) - block_size - 1, (32,))

    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])

    return x, y

In [7]:
class LSTMLanguageModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=128, num_layers=2):

        super().__init__()

        # Convert character IDs into vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # LSTM layers
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        # Predict next character
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):

        # Character embedding
        x = self.embedding(x)

        # LSTM processing
        out, hidden = self.lstm(x, hidden)

        # Predict character probabilities (logits)
        logits = self.fc(out)

        return logits, hidden

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMLanguageModel(vocab_size).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 2000

for step in range(epochs):

    model.train()

    x, y = get_batch("train")

    x, y = x.to(device), y.to(device)

    # Forward propagation
    logits, hidden = model(x)

    # Calculate loss
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        y.reshape(-1)
    )

    # Backpropagation
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 200 == 0:
        print(f"Step: {step}, Loss: {loss.item():.4f}")

Step: 0, Loss: 4.1312
Step: 200, Loss: 2.2562
Step: 400, Loss: 1.9776
Step: 600, Loss: 1.8378
Step: 800, Loss: 1.7186
Step: 1000, Loss: 1.7256
Step: 1200, Loss: 1.6567
Step: 1400, Loss: 1.5382
Step: 1600, Loss: 1.6047
Step: 1800, Loss: 1.4889


In [11]:
def generate_text(model, start_text, max_new_tokens=500, temperature=0.8):

    model.eval()

    # Convert starting text into numbers
    input_ids = [stoi.get(c, stoi.get(" ", 0)) for c in start_text]

    idx = torch.tensor([input_ids], dtype=torch.long).to(device)

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # Use the last 100 characters
            idx_cond = idx[:, -block_size:]

            # Predict next character
            logits, hidden = model(idx_cond)

            logits = logits[:, -1, :] / temperature

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Select next character
            next_char = torch.multinomial(probs, num_samples=1)

            # Append predicted character
            idx = torch.cat((idx, next_char), dim=1)

    # Convert numbers back to characters
    result = "".join([itos[i] for i in idx[0].tolist()])

    return result


# Generate text
print(generate_text(model, "ROMEO:", max_new_tokens=500))

ROMEO:
A more and sale him,
And you live and patces proper:
Which mother infire one fare to the remail a mitter.

BRUTUS:
That to this most news.

Second Servingman:
Not our grave consul, were then this now and
Look. You, as say pressong patch.

First Senator:
The barts which be unnedicition. But he shall the thow
My much shall a letter man but to
A joil thou art man it you.

Second Carin:
Come, therefore thy hamber the gods.

QUEEN ELIZABETH:
What cannot not mea.

BRUTUS:
And you.

AUFIDIUS:
The more


In [13]:
print(generate_text(model, "JULIET:", 500))
print(generate_text(model, "KING:", 500))
print(generate_text(model, "ROMEO:", 500))

JULIET:
The might be the beconted show this was well to breathed pate,
Than could to the remite.

MENENIUS:
The braises of plores. Which he shour better three
The ower to bluse in this not; you mavy,
Their did enemies with the warth, sir!
Wall fear for no more spean of sir; thought I would be done
And love with love the last;
And Consious monedly, and your devended me beat
Be a glant curdest I thank the gods well, which being, and I speak,
St have not putt not and like by did father.

LADY ANNE:
Come; 
KING: I'll may find,
That are they are else.

All:
Now, you hast mens, and there a good for he lid word,--

Third Citizen:
Nother to can frempred, the stlank the senfight endious to his former, have wo'd, best to them most fair.
Well are I can net come.

VOLUMNIA:
They will a bad. Corinous: and I would hear.

BRUTUS:
And the cappition will beliest my fit,
I state not in make the desup,
And as the cannot them to have dort
The shall empries, and that all them
which such ament and truth's t

In [17]:
torch.save(model.state_dict(), "small_lstm_gpt.pth")

In [19]:
model = LSTMLanguageModel(vocab_size).to(device)
model.load_state_dict(torch.load("small_lstm_gpt.pth", map_location=device))
model.eval()

LSTMLanguageModel(
  (embedding): Embedding(62, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=128, out_features=62, bias=True)
)